In [2]:
!pip install seaborn

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [4]:
cleaned_cc = pd.read_csv('cleaned_cc_calls_file.csv')

In [5]:
coref=pd.read_csv('match3.csv')

In [6]:
cleaned_cc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31519 entries, 0 to 31518
Data columns (total 12 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Co_Ref                                    31519 non-null  object 
 1   Contact_ID                                31519 non-null  float64
 2   Call_Date                                 31519 non-null  object 
 3   Direction                                 31519 non-null  object 
 4   cc_business_struggles_financial_hardship  31519 non-null  object 
 5   cc_dissatisfaction_support                31519 non-null  object 
 6   cc_contractor_sentiment                   31519 non-null  object 
 7   cc_contractor_sentiment_issues_score      31519 non-null  object 
 8   cc_pricing_sentiment_impact               31519 non-null  object 
 9   cc_refund_discussed                       31519 non-null  object 
 10  cc_contractor_complained          

In [7]:
cleaned_cc['Call_Date']=pd.to_datetime(cleaned_cc['Call_Date'],format='%Y-%m-%d')
coref['Prospect_Renewal_Date'] = pd.to_datetime(
    coref['Prospect_Renewal_Date'],
    dayfirst=True,
    errors='coerce'
)

In [8]:
cleaned_cc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31519 entries, 0 to 31518
Data columns (total 12 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Co_Ref                                    31519 non-null  object        
 1   Contact_ID                                31519 non-null  float64       
 2   Call_Date                                 31519 non-null  datetime64[ns]
 3   Direction                                 31519 non-null  object        
 4   cc_business_struggles_financial_hardship  31519 non-null  object        
 5   cc_dissatisfaction_support                31519 non-null  object        
 6   cc_contractor_sentiment                   31519 non-null  object        
 7   cc_contractor_sentiment_issues_score      31519 non-null  object        
 8   cc_pricing_sentiment_impact               31519 non-null  object        
 9   cc_refund_discussed         

In [9]:
merged=coref.merge(cleaned_cc,on='Co_Ref',how='left')

In [10]:
from datetime import timedelta 
mask = (
    ((merged['Call_Date'] >= merged['Prospect_Renewal_Date'] - timedelta(days=45)) &
     (merged['Call_Date'] <= merged['Prospect_Renewal_Date'] - timedelta(days=14))) |
    merged['Contact_ID'].isnull()
)
result = merged[mask]

In [11]:
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 76885 entries, 0 to 161370
Data columns (total 14 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Co_Ref                                    76885 non-null  object        
 1   Prospect_Outcome                          76885 non-null  object        
 2   Prospect_Renewal_Date                     76885 non-null  datetime64[ns]
 3   Contact_ID                                4097 non-null   float64       
 4   Call_Date                                 4097 non-null   datetime64[ns]
 5   Direction                                 4097 non-null   object        
 6   cc_business_struggles_financial_hardship  4097 non-null   object        
 7   cc_dissatisfaction_support                4097 non-null   object        
 8   cc_contractor_sentiment                   4097 non-null   object        
 9   cc_contractor_sentiment_issues_s

In [12]:
duplicate_counts = (
    result.groupby(['Co_Ref', 'Call_Date'])
         .size()
         .reset_index(name='count')
)
print (len(duplicate_counts))

3922


In [13]:
duplicates_only = duplicate_counts[duplicate_counts['count'] > 1]

In [14]:
len(duplicates_only)

164

In [15]:
df = result.copy()

df_non_null = df[df['Call_Date'].notna()]
df_null = df[df['Call_Date'].isna()]

In [16]:
df_non_null = df_non_null.sort_values(
    by=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date'],
    ascending=[True, True, False]
)

In [17]:
def resolve_same_call_date(group):
    if len(group) == 1:
        return group.iloc[0]
    return group.mode().iloc[0]

In [18]:
df_non_null = (
    df_non_null.groupby(['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date'], as_index=False)
               .apply(resolve_same_call_date)
               .reset_index(drop=True)
)

C:\Users\ranji\AppData\Local\Temp\ipykernel_10120\3800539238.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(resolve_same_call_date)


In [19]:
df_non_null = (
    df_non_null.sort_values(by='Call_Date', ascending=False)
               .drop_duplicates(subset=['Co_Ref', 'Prospect_Renewal_Date'])
)

In [20]:
df_null = (
    df_null.groupby(['Co_Ref', 'Prospect_Renewal_Date'], as_index=False)
           .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
)

In [21]:
final_df = pd.concat([df_non_null, df_null], ignore_index=True)

In [22]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75925 entries, 0 to 75924
Data columns (total 14 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Co_Ref                                    75925 non-null  object        
 1   Prospect_Outcome                          75925 non-null  object        
 2   Prospect_Renewal_Date                     75925 non-null  datetime64[ns]
 3   Contact_ID                                3137 non-null   float64       
 4   Call_Date                                 3137 non-null   datetime64[ns]
 5   Direction                                 3137 non-null   object        
 6   cc_business_struggles_financial_hardship  3137 non-null   object        
 7   cc_dissatisfaction_support                3137 non-null   object        
 8   cc_contractor_sentiment                   3137 non-null   object        
 9   cc_contractor_sentiment_issu

In [23]:
final_df.to_csv('final_df.csv', index=False)

In [24]:
new_df=final_df[['Co_Ref','Prospect_Renewal_Date','cc_business_struggles_financial_hardship','cc_contractor_sentiment','cc_contractor_sentiment_issues_score']]

In [25]:
display(new_df.head())

,Co_Ref,Prospect_Renewal_Date,cc_business_struggles_financial_hardship,cc_contractor_sentiment,cc_contractor_sentiment_issues_score
0,ME8032,2026-02-12,No,Neutral,Not Discussed
1,IQ7300,2026-02-19,No,Satisfied,Not Discussed
2,WZ5097,2026-01-23,No,Neutral,80
3,JZ3557,2026-02-11,No,Satisfied,Not Discussed
4,RV0970,2026-02-14,No,Satisfied,85


In [26]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75925 entries, 0 to 75924
Data columns (total 5 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Co_Ref                                    75925 non-null  object        
 1   Prospect_Renewal_Date                     75925 non-null  datetime64[ns]
 2   cc_business_struggles_financial_hardship  3137 non-null   object        
 3   cc_contractor_sentiment                   3137 non-null   object        
 4   cc_contractor_sentiment_issues_score      3137 non-null   object        
dtypes: datetime64[ns](1), object(4)
memory usage: 2.9+ MB


In [27]:
new_df.to_csv('new_df_01.csv', index=False)        

In [3]:
bills=pd.read_csv('billings_cleaned.csv')

In [29]:
bills.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113894 entries, 0 to 113893
Data columns (total 27 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Co_Ref                      113894 non-null  object 
 1   Sustainability_Score        113894 non-null  float64
 2   Total_Renewal_Score_New     113894 non-null  float64
 3   Auto_Renewal_Score          113894 non-null  int64  
 4   Status_Scores               113894 non-null  int64  
 5   Anchoring_Score             113894 non-null  float64
 6   Tenure_Scores               113894 non-null  float64
 7   Renewal_Score_At_Release    113894 non-null  float64
 8   Proforma_Account_Stage      113894 non-null  object 
 9   Current_Auto_Renewal_Flag   113894 non-null  object 
 10  Current_World_Pay_Token     113894 non-null  object 
 11  Proforma_Membership_Status  113894 non-null  object 
 12  Prospect_Status             113894 non-null  object 
 13  Band          

In [8]:
new_billings=bills[['Co_Ref','Prospect_Renewal_Date','Proforma_Account_Stage','Proforma_Membership_Status','Status_Scores','Sustainability_Score','Tenure_Scores','Band']]

In [5]:
new_billings.info()

<class 'pandas.DataFrame'>
RangeIndex: 113894 entries, 0 to 113893
Data columns (total 7 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Co_Ref                      113894 non-null  str    
 1   Prospect_Renewal_Date       113894 non-null  str    
 2   Proforma_Account_Stage      113894 non-null  str    
 3   Proforma_Membership_Status  113894 non-null  str    
 4   Status_Scores               113894 non-null  int64  
 5   Sustainability_Score        113894 non-null  float64
 6   Tenure_Scores               113894 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.1 MB


In [10]:
display(new_billings)

,Co_Ref,Prospect_Renewal_Date,Proforma_Account_Stage,Proforma_Membership_Status,Status_Scores,Sustainability_Score,Tenure_Scores,Band
0,VT6174,05-11-2024,Published,Accredited,9,8.0,9.0,Band C1
1,VD3828,09-08-2025,Published,Accredited,9,8.0,8.0,Band C1
2,DV8120,12-03-2025,Membership Only,Member Only,0,8.0,9.5,Band C1
3,EZ9894,29-06-2025,Renewal Process,Accredited,9,9.5,9.5,Band C1
4,FA8957,25-03-2025,Published,Accredited,8,9.5,8.5,Band C1
...,...,...,...,...,...,...,...,...
113889,VI6211,25-04-2023,Published,Accredited,0,8.0,9.0,Band B
113890,NI7208,24-06-2023,Published,Accredited,0,8.0,7.0,Band B
113891,DO6023,15-09-2023,Published,Accredited,0,8.0,8.5,Band B
113892,KR5815,03-08-2023,Published,Accredited,7,8.0,9.0,Band B


In [9]:
new_billings.to_csv('new_billings.csv', index=False)

In [34]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75925 entries, 0 to 75924
Data columns (total 5 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Co_Ref                                    75925 non-null  object        
 1   Prospect_Renewal_Date                     75925 non-null  datetime64[ns]
 2   cc_business_struggles_financial_hardship  3137 non-null   object        
 3   cc_contractor_sentiment                   3137 non-null   object        
 4   cc_contractor_sentiment_issues_score      3137 non-null   object        
dtypes: datetime64[ns](1), object(4)
memory usage: 2.9+ MB
